In [37]:
import polars as pl
import numpy as np

data = pl.read_csv("./Source/returns.csv")
data.columns = ["Date", "Cash", "Bonds", "Equities", "CPI"]
data = data.with_columns(pl.col("Date").str.to_date())

In [55]:
# annual returns
data_annual = (data
    .unpivot(index="Date")
    .sort("Date")
    .with_columns(pl.col("Date").dt.year().alias("Year"))
    .group_by("variable", "Year", maintain_order=True)
    .agg(((pl.col("value") + 1).product() -1).alias("AnnualReturn"))
    .pivot(index="Year", on="variable", values="AnnualReturn")
)
covariance = (data_annual.select("Cash", "Bonds", "Equities", "CPI").to_pandas().cov())

# write out covariance matrix
covariance.to_csv("./Final/covariance.csv", index=True)
covariance

,Cash,Bonds,Equities,CPI
Cash,0.000748,0.001079,-0.000116,0.000014
Bonds,0.001079,0.003270,-0.000151,-0.000260
Equities,-0.000116,-0.000151,0.019288,-0.000513
CPI,0.000014,-0.000260,-0.000513,0.000257


In [56]:
data_annual.select("Cash", "Bonds", "Equities", "CPI").to_pandas().mean()

Cash        0.042366
Bonds       0.069303
Equities    0.105462
CPI         0.023551
dtype: float64